# **SI 313 WN26: Final Project Progress Report**
##### Nick Pisarczyk - Monday, 04/13/26<br>uniqname: npisar<br> 
---

# **Progress Summary**
aaa

---

### Project Plan / Ethics Review
https://docs.google.com/document/d/1MV13jZvPOOKMSl6CHwO51uBc4c8MzMOqv2vkrDHoB1g/edit?usp=sharing

---

# **311 Data Gathering / Cleaning**

In [386]:
import requests
import pandas as pd
import re

In [387]:
# helper function to preview dfs
def preview_df(df, head=False, num=20):
    print(f"{'='*50}")
    print(f"Columns:\n{df.columns.tolist()[0:]}")
    print(f"{'='*50}")
    print(f"Shape:\n{df.shape}")
    print(f"{'='*50}")
    if head:
        print(f"Data Head:\n{df.head(num)}")
        print(f"{'='*50}")

### **WPRDC 311 data from URL endpoint / save to .csv**
Grab the WPRDC 311 data from the URL endpoint

In [388]:
# # data from https://data.wprdc.org/dataset/311-data/resource/29462525-62a6-45bf-9b5e-ad2e1c06348d
# # using a SQL query to get data from just 2024

# url = (
#     "https://data.wprdc.org/api/action/datastore_search_sql"
#     "?sql=SELECT * from \"29462525-62a6-45bf-9b5e-ad2e1c06348d\" "
#     "WHERE create_date_utc >= '2024-01-01' AND create_date_utc < '2025-01-01'"
# )

# resp = requests.get(url, timeout=60)
# resp.raise_for_status()
# data = resp.json()

# if not data.get("success"):
#     raise RuntimeError(data)

# records = data["result"]["records"]
# df_long = pd.DataFrame.from_records(records)

# df_long.to_csv("data/wprdc_2024.csv", index=False)
# print(f"Wrote {len(df)} rows to wprdc_2024.csv")

### **Open the saved csv**
Instead of calling the url every time, use the saved .csv to gather data

In [389]:
csv_path = "data/wprdc_2024.csv"
df_long = pd.read_csv(csv_path)

preview_df(df_long, head=False, num=20)

Columns:
['_id', '_full_text', 'group_id', 'num_requests', 'parent_closed', 'status_name', 'status_code', 'dept', 'request_type_name', 'request_type_id', 'create_date_et', 'create_date_utc', 'last_action_et', 'last_action_utc', 'closed_date_et', 'closed_date_utc', 'origin', 'street', 'cross_street', 'street_id', 'cross_street_id', 'city', 'neighborhood', 'census_tract', 'council_district', 'ward', 'police_zone', 'latitude', 'longitude', 'geo_accuracy']
Shape:
(32000, 30)


### **311 Data Cleaning**
Get rid of redundant columns, rename columns, create new df to store this in

In [390]:
keep_cols = ["_id", "create_date_utc", "request_type_name", "census_tract", "neighborhood"]
keep_cols = [c for c in keep_cols if c in df_long.columns]

df_small = df_long.copy()
df_small = df_small.loc[:, keep_cols]

# get rid of columns we don't want
DROP_IF_PRESENT = [
    "street", "cross_street", "latitude", "longitude",
]
df_small = df_small.drop(columns=[c for c in DROP_IF_PRESENT if c in df_small.columns], errors="ignore")

# column names
rename_map = {"_id":"id", "create_date_utc":"created_date", "request_type_name":"request"}
df_small = df_small.rename(columns=rename_map)

preview_df(df_small, head=True, num=20)

Columns:
['id', 'created_date', 'request', 'census_tract', 'neighborhood']
Shape:
(32000, 5)
Data Head:
        id         created_date                   request  census_tract  \
0   796260  2024-12-05T13:07:00  Missed Recycling Pick Up  4.200308e+10   
1   784402  2024-10-01T19:45:00              Weeds/Debris           NaN   
2    61204  2024-09-01T22:52:00           Illegal Parking  4.200398e+10   
3   656756  2024-07-08T17:17:00            Rodent control  4.200318e+10   
4   625758  2024-01-11T14:18:00       Pruning (city tree)           NaN   
5   785802  2024-10-08T19:38:00              City Website           NaN   
6   800014  2024-12-17T20:06:00      Catch Basin, Clogged           NaN   
7    77784  2024-08-21T19:38:00              Weeds/Debris           NaN   
8   611835  2024-05-06T15:56:00   Litter, Public Property           NaN   
9   786247  2024-10-09T16:09:00  Missed Recycling Pick Up  4.200305e+10   
10   61211  2024-04-23T15:16:00    Curb paint application           NaN

#### **More cleaning...**

In [391]:
# Drop entries from df_small that have NO neighborhood AND no census tract
df_small = df_small.copy().dropna(subset=['neighborhood', 'census_tract'], how='all')

# load neighborhood / tract index
df_index = pd.read_csv('data/index_pittsburghneighborhoods_blocks_2020.csv')



# standardize for neighborhood / tract data bridge
df_index['tract_geoid'] = '42003' + df_index['TRACT'].astype(str).str.zfill(6)
neigh_to_tract = df_index.groupby('Neighborhood')['tract_geoid'].first().to_dict()
# print(f"Neighborhood - Tract mapping:")
# for k, v in neigh_to_tract.items():
#     print(f"    {k}: {neigh_to_tract[k]}")

# more standardization
    # converts rows with tract_geoid values into strings, or None if empty
df_small['tract_geoid'] = df_small['census_tract'].apply(lambda x: str(int(x)) if pd.notnull(x) else None)



# clean some neighborhood names
neighborhood_names_fix = {
    'Arlington': 'Arlington - Arlington Heights', 
    'Arlington Heights': 'Arlington - Arlington Heights',
    'Mount Oliver Borough': 'Mt. Oliver'
}
df_small['neighborhood_clean'] = df_small['neighborhood'].replace(neighborhood_names_fix)



# fill missing tracts using the neighborhood bridge
for index, row in df_small.iterrows():
    if pd.isna(row['tract_geoid']):
        name = row['neighborhood_clean']
        mapped_tract = neigh_to_tract.get(name)
        if mapped_tract:
            df_small.at[index, 'tract_geoid'] = mapped_tract

# avoid duplicating 311 requests by neighborhood block
# search on geoid / tract to drop any duplicates
df_index_unique = df_index[['tract_geoid', 'TRACT']].drop_duplicates(subset=['tract_geoid'])

df_joined = df_small.merge(
    df_index_unique, 
    on='tract_geoid', 
    how='inner'
)



# normalize tracts to be their normal values, not * 100 like stored in data
# convert any integers with trailing .0 into ints
df_joined['tract'] = df_joined['TRACT'] / 100
df_joined['tract'] = df_joined['tract'].apply(lambda x: f"{int(x)}" if x % 1 == 0 else f"{x}")



# final cleanup
    # flip next two lines if want full tract value not divided by 100 (for whatever reason)
df_joined = df_joined.drop(columns="TRACT")
# df_joined = df_joined.rename(columns={'TRACT': 'tract_un-normal'})
final_cols = ['id', 'created_date', 'request', 'neighborhood', 'tract']
df_joined = df_joined[final_cols]



# if needed, save the final mapped dataset
# df_joined.to_csv('data/df_joined.csv', index=False)
print(f"Successfully joined 311 data!")

Successfully joined 311 data!


### **Load ACS data and combine median income with 311 data**

In [392]:
# helper function for grabbing tracts from headers in ACS data
def extract_tract(header):
    match = re.search(r'Census Tract ([\d\.]+)', header)
    return match.group(1) if match else None

In [393]:
# load ACS data
df_acs = pd.read_csv('data/ACS-PA-5Y-2024_DP03.csv')

# need to match the "wide" ACS data with the "long" 311 data
# .melt needed because every tract is its own column in ACS.
# need all these tracts to be rows in df_final to match with df_joined
income_row = df_acs[df_acs['Label (Grouping)'].str.contains("Median household income", na=False)]
df_income = income_row.melt(id_vars='Label (Grouping)', var_name='header', value_name='median_income')



# only grab !!Estimate columns (they actually have the data here)
df_income = df_income[df_income['header'].str.endswith('!!Estimate')].copy()
# print(f"{df_income}")

# extract the tract numbers from the header with re
# e.g. '103.01' from 'Census Tract 103.01; Allegheny County...'
df_income['tract_acs'] = df_income['header'].apply(extract_tract)

df_income['median_income'] = (
    df_income['median_income']
    .str.replace(',', '', regex=False)
    .replace('-', None)
    .astype(float)
)
# print(f"{df_income}")




# merge with df_joined
df_joined['tract'] = df_joined['tract'].astype(str)
df_income['tract_acs'] = df_income['tract_acs'].astype(str)

df_all = df_joined.merge(
    df_income[['tract_acs', 'median_income']], 
    left_on='tract', 
    right_on='tract_acs', 
    how='left'
).drop(columns=['tract_acs'])




# remove any rows with NaN in the median_income
rows_before = len(df_all)
neighborhoods_before = set(df_all['neighborhood'].unique())
# save the neighborhoods that get removed
df_missing = df_all[df_all['median_income'].isna()]
neighborhoods_with_missing = set(df_missing['neighborhood'].unique())

# create final df
df_final = df_all.dropna(subset=['median_income']).copy()

# what neighborhoods were removed entirely?
rows_after = len(df_final)
neighborhoods_after = set(df_final['neighborhood'].unique())
neighborhoods_fully_removed = neighborhoods_before - neighborhoods_after




print("--- ANALYSIS SUMMARY ---")
print(f"Total rows removed: {rows_before - rows_after}")
print(f"Neighborhoods completely removed from analysis: {list(neighborhoods_fully_removed) if neighborhoods_fully_removed else 'None'}")
print(f"Neighborhoods with at least some missing tract data: {list(neighborhoods_with_missing)}")
print(f"\n\n\n")

--- ANALYSIS SUMMARY ---
Total rows removed: 1221
Neighborhoods completely removed from analysis: ['Chateau']
Neighborhoods with at least some missing tract data: ['Highland Park', 'Perry North', 'South Oakland', 'Squirrel Hill North', 'Bluff', 'Allegheny West', 'Point Breeze', 'Squirrel Hill South', 'Manchester', nan, 'Chateau', 'North Shore', 'Central Oakland', 'North Oakland']






### **Preview cleaned data**

In [416]:
# preview the new df
preview_df(df_final, head=False, num=20)
df_final

Columns:
['id', 'created_date', 'request', 'neighborhood', 'tract', 'median_income']
Shape:
(25153, 6)


,id,created_date,request,neighborhood,tract,median_income
0,796260,2024-12-05T13:07:00,Missed Recycling Pick Up,Bloomfield,804,65116.0
1,784402,2024-10-01T19:45:00,Weeds/Debris,North Oakland,404,60192.0
3,656756,2024-07-08T17:17:00,Rodent control,Allentown,1803,43125.0
4,625758,2024-01-11T14:18:00,Pruning (city tree),North Oakland,404,60192.0
5,800014,2024-12-17T20:06:00,"Catch Basin, Clogged",North Oakland,404,60192.0
...,...,...,...,...,...,...
26369,733675,2024-05-08T18:40:00,Tree Removal,Homewood West,1307,31799.0
26370,733754,2024-05-06T13:26:00,Commercial Refuse/Dumpsters,East Hills,1306,32083.0
26371,814675,2024-11-05T14:46:00,Commercial Refuse/Dumpsters,Homewood North,1302,50605.0
26372,733684,2024-09-20T15:07:00,Sinkhole,Homewood North,1302,50605.0


In [415]:
# preview n/t mapping
nt_map = df_final.groupby('neighborhood')['tract'].first().to_dict()
ni_map = df_final.groupby('neighborhood')['median_income'].first().to_dict()
# print(nt_map)
# print(ni_map)

neighborhood_info_map = {}
for neighborhood, tract in nt_map.items():
    income = ni_map.get(neighborhood)
    neighborhood_info_map[neighborhood] = tract, income

i = 1
print(f"      Neighborhood  |  Tract  |  Median Income Mapping:")
for k, v in neighborhood_info_map.items():
    if i < 10:
        print(f"{i}     {k}: {v}")
    else:
        print(f"{i}    {k}: {v}")
    i+=1

      Neighborhood  |  Tract  |  Median Income Mapping:
1     Allegheny Center: ('5627', 60658.0)
2     Allegheny West: ('5627', 60658.0)
3     Allentown: ('1803', 43125.0)
4     Arlington: ('1608', 55761.0)
5     Arlington Heights: ('1608', 55761.0)
6     Banksville: ('2023', 75808.0)
7     Bedford Dwellings: ('509', 11986.0)
8     Beechview: ('1916', 70436.0)
9     Beltzhoover: ('5624', 41098.0)
10    Bloomfield: ('804', 65116.0)
11    Bluff: ('402', 31042.0)
12    Bon Air: ('5624', 41098.0)
13    Brighton Heights: ('2701', 75990.0)
14    Brookline: ('3206', 82537.0)
15    California-Kirkbride: ('5652', 92500.0)
16    Carrick: ('2901', 47143.0)
17    Central Business District: ('201', 82409.0)
18    Central Lawrenceville: ('901', 112885.0)
19    Central Northside: ('5651', 68845.0)
20    Central Oakland: ('405', 16721.0)
21    Crafton Heights: ('2814', 37750.0)
22    Crawford-Roberts: ('201', 82409.0)
23    Duquesne Heights: ('1911', 104286.0)
24    East Allegheny: ('2620', 38561.0)


---

# **Data Exploration**

,id,created_date,request,neighborhood,tract,median_income
0,796260,2024-12-05T13:07:00,Missed Recycling Pick Up,Bloomfield,804,65116.0
1,784402,2024-10-01T19:45:00,Weeds/Debris,North Oakland,404,60192.0
3,656756,2024-07-08T17:17:00,Rodent control,Allentown,1803,43125.0
4,625758,2024-01-11T14:18:00,Pruning (city tree),North Oakland,404,60192.0
5,800014,2024-12-17T20:06:00,"Catch Basin, Clogged",North Oakland,404,60192.0
...,...,...,...,...,...,...
26369,733675,2024-05-08T18:40:00,Tree Removal,Homewood West,1307,31799.0
26370,733754,2024-05-06T13:26:00,Commercial Refuse/Dumpsters,East Hills,1306,32083.0
26371,814675,2024-11-05T14:46:00,Commercial Refuse/Dumpsters,Homewood North,1302,50605.0
26372,733684,2024-09-20T15:07:00,Sinkhole,Homewood North,1302,50605.0
